In [8]:
from langchain_ollama import OllamaEmbeddings  
import numpy as np

In [9]:
sentences = [
    "The sun rises in the east and sets in the west.",
    "Photosynthesis is the process by which plants make food using sunlight.",
    "Dogs are loyal animals and make great companions.",
    "The Earth revolves around the Sun once every 365 days.",
    "Neural networks are inspired by the human brain structure.",
    "Water boils at 100 degrees Celsius at sea level.",
    "Machine learning models learn patterns from training data.",
    "The Amazon rainforest produces 20% of the world's oxygen.",
    "Python is a popular programming language for data science.",
    "Gravity pulls objects toward the center of the Earth.",
]

# ─────────────────────────────────────────────────────────────────
# Load the Ollama embedding model
# ─────────────────────────────────────────────────────────────────

print("🔄 Loading embedding model...")
embedder = OllamaEmbeddings(model="mxbai-embed-large")

# ─────────────────────────────────────────────────────────────────
# STEP 3: Embed all sentences
# ─────────────────────────────────────────────────────────────────

print(" Embedding all sentences...\n")
sentence_vectors = embedder.embed_documents(sentences)

# just to see embedding how they look
print(f"✅ Each sentence is now a vector of {len(sentence_vectors[0])} numbers")
print(f"   Example (first 5 values of sentence 1): {sentence_vectors[0][:5]}\n")

🔄 Loading embedding model...
 Embedding all sentences...

✅ Each sentence is now a vector of 1024 numbers
   Example (first 5 values of sentence 1): [-0.01757053, 0.020696158, -0.023339357, -0.059489645, -0.05234188]



In [ ]:
# ─────────────────────────────────────────────────────────────────
# Define cosine similarity
#
# Cosine similarity measures the ANGLE between two vectors.
# Score = 1.0  → identical meaning
# Score = 0.8+ → very related
# Score = 0.5  → somewhat related
# Score = 0.0  → completely unrelated
#
# Formula: cos(θ) = (A · B) / (|A| × |B|)
#   A · B  = dot product (how much they point in the same direction)
#   |A|    = magnitude (length of vector A)
# ─────────────────────────────────────────────────────────────────

def cosine_similarity(vec_a, vec_b):
    a = np.array(vec_a)
    b = np.array(vec_b)
    dot_product  = np.dot(a, b)          # how aligned are they?
    magnitude_a  = np.linalg.norm(a)     # length of vector A
    magnitude_b  = np.linalg.norm(b)     # length of vector B
    return dot_product / (magnitude_a * magnitude_b)

# ─────────────────────────────────────────────────────────────────
# Ask a question and find similar sentences
#
# We embed the QUESTION using the same model.
# Then compare it against every sentence using cosine similarity.
# The highest score = most relevant sentence to the question.
#
# This is EXACTLY what a vector database like ChromaDB does
# internally during RAG retrieval!
# ─────────────────────────────────────────────────────────────────

def ask(question: str):
    print(f"\n{'='*60}")
    print(f"❓ Question: {question}")
    print(f"{'='*60}")

    # Embed the question — same model, same vector space
    question_vector = embedder.embed_query(question)

    # Compare question vector vs every sentence vector
    scores = []
    for i, sent_vector in enumerate(sentence_vectors):
        score = cosine_similarity(question_vector, sent_vector)
        scores.append((score, sentences[i]))

    # Sort by score — highest similarity first
    scores.sort(reverse=True)

    # Print results with a simple visual bar
    print(f"\n{'Rank':<5} {'Score':<10} {'Bar':<25} Sentence")
    print("-" * 90)
    for rank, (score, sentence) in enumerate(scores, 1):
        bar   = "█" * int(score * 30)        # visual bar scaled to 30 chars
        short = sentence[:55] + "..." if len(sentence) > 55 else sentence
        print(f"{rank:<5} {score:<10.4f} {bar:<25} {short}")

    print(f"\n🏆 Most relevant: \"{scores[0][1]}\"")

# ─────────────────────────────────────────────────────────────────
# some questions to test our simple RAG retrieval
# ─────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    ask("How do plants use sunlight?")
    ask("Tell me about machine learning and AI")
    ask("What happens when water gets very hot?")



❓ Question: How do plants use sunlight?

Rank  Score      Bar                       Sentence
------------------------------------------------------------------------------------------
1     0.8330     ████████████████████████  Photosynthesis is the process by which plants make food...
2     0.5224     ███████████████           The sun rises in the east and sets in the west.
3     0.4777     ██████████████            The Earth revolves around the Sun once every 365 days.
4     0.4701     ██████████████            The Amazon rainforest produces 20% of the world's oxyge...
5     0.3842     ███████████               Gravity pulls objects toward the center of the Earth.
6     0.3781     ███████████               Machine learning models learn patterns from training da...
7     0.3585     ██████████                Neural networks are inspired by the human brain structu...
8     0.3507     ██████████                Water boils at 100 degrees Celsius at sea level.
9     0.3260     █████████   